In [ ]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

In [ ]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

In [ ]:
from nanover.imd.imd_state import INTERACTION_PREFIX, ParticleInteraction

RESTRAINT_PREFIX = f"{INTERACTION_PREFIX}MOVEABLE-RESTRAINT."

MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
ACTIVE_RESTRAINTS: set[str] = set()

def add_restraint(indexes):
    key = f"{RESTRAINT_PREFIX}{len(MOVEABLE_RESTRAINTS)}"
    restraint = ParticleInteraction((0, 0, 0), indexes)
    restraint.scale = 1000
    restraint.interaction_type = "spring"
    restraint.reset_velocities = True
    MOVEABLE_RESTRAINTS[key] = restraint
    # enable_restraint(key)
    return key


def enable_restraint(key):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    print("ENABLE", key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = positions[restraint.particles].sum(axis=0) / len(restraint.particles)
    imd.insert_interaction(key, restraint)


def disable_restraint(key):
    if key not in ACTIVE_RESTRAINTS:
        return
    print("DISABLE", key)
    ACTIVE_RESTRAINTS.remove(key)
    imd.remove_interaction(key)


def on_interaction_started(*, key: str, interaction: ParticleInteraction):
    if RESTRAINT_PREFIX in key:
        return

    interacted_indexes = set(interaction.particles)
    for key, restraint in MOVEABLE_RESTRAINTS.items():
        if interacted_indexes.intersection(restraint.particles):
            disable_restraint(key)


def on_interaction_stopped(*, key: str, interaction: ParticleInteraction):
    if RESTRAINT_PREFIX in key:
        return

    interacted_indexes = set(interaction.particles)
    for key, restraint in MOVEABLE_RESTRAINTS.items():
        if interacted_indexes.intersection(restraint.particles):
            enable_restraint(key)


imd_runner.app_server.imd.interaction_started.add_callback(on_interaction_started)
imd_runner.app_server.imd.interaction_stopped.add_callback(on_interaction_stopped)

In [ ]:
from nanover.websocket import NanoverImdClient

renderer = {
    "color": "blue",
    "scale": 0.1,
    "render": "liquorice",
}

with NanoverImdClient.from_runner(imd_runner) as client:
    for index in [0, 5, 8, 11, 16]:
        residue = universe.residues[index]
        indices = [int(i) for i in residue.atoms.indices]
        add_restraint(indices)

        with client.create_selection(f"residue.{index}", indices).modify() as selection:
            selection.renderer = renderer
            selection.velocity_reset = True
            selection.interaction_method = "group"
